# Data Restructuring

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/feature_engineered_data.csv")
df.head()

,LOG_ID,MRN,SMRTDTA_ELEM_VALUE,AGE,HEIGHT,WEIGHT,SEX,AN_START_DATETIME,Lab Code,Lab Name,Observation Value,Measurement Units,Collection Datetime,hypoxemia,bmi,sex
0,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,1988-5,C reactive protein,7.6,MG/DL,2021-08-31 05:44:00,0,24.170546,1
1,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,2028-9,Carbon dioxide,23.0,mmol/L,2021-09-02 06:30:00,0,24.170546,1
2,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,2345-7,Glucose,119.0,mg/dL,2021-09-02 06:30:00,0,24.170546,1
3,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,20570-8,Hematocrit,27.8,%,2021-09-01 07:50:00,0,24.170546,1
4,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,30350-3,Hemoglobin,9.2,G/DL,2021-09-01 07:50:00,0,24.170546,1


#### Reshape lab data from long to wide format

In [3]:
# rename the lab names

lab_name_map = {
    "C reactive protein": "crp",
    "Carbon dioxide": "co2",
    "Glucose": "glucose",
    "Hematocrit": "hematocrit",
    "Hemoglobin": "hemoglobin",
    "Leukocytes^^corrected for nucleated erythrocytes": "leukocytes",
    "Potassium": "potassium",
    "Sodium": "sodium",
    "pH": "pH",
    "Lactate": "lactate"
}

df["lab"] = df["Lab Name"].map(lab_name_map)

In [4]:
# pivot lab values to wide format

value_wide = df.pivot(
    index=["LOG_ID", "MRN"],
    columns="lab",
    values="Observation Value"
).reset_index()

value_wide.head()

lab,LOG_ID,MRN,co2,crp,glucose,hematocrit,hemoglobin,lactate,leukocytes,pH,potassium,sodium
0,0004678c0906bb9e,6b03d87f76b29311,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0
1,0011abe1d4cc9915,c28fd59799fdb891,24.0,NaN,121.0,26.7,8.4,NaN,8.0,NaN,4.2,141.0
2,00126ebdaa3f286a,f3ef9a2e287a78e8,27.0,15.9,106.0,36.7,12.4,NaN,9.8,7.42,3.7,141.0
3,0013fc9808033c8f,ef268d8abece3181,26.0,NaN,122.0,39.5,14.0,NaN,5.7,NaN,4.5,137.0
4,001656cc37a919fd,73ae956471950fd0,23.0,0.3,96.0,38.7,12.9,NaN,8.4,NaN,3.8,138.0


In [5]:
# pivot lab units to wide format

unit_wide = df.pivot(
    index=["LOG_ID", "MRN"],
    columns="lab",
    values="Measurement Units"
).reset_index()

unit_wide.head()

lab,LOG_ID,MRN,co2,crp,glucose,hematocrit,hemoglobin,lactate,leukocytes,pH,potassium,sodium
0,0004678c0906bb9e,6b03d87f76b29311,mmol/L,MG/DL,mg/dL,%,G/DL,NaN,THOUS/MCL,NaN,mmol/L,mmol/L
1,0011abe1d4cc9915,c28fd59799fdb891,mmol/L,NaN,mg/dL,%,G/DL,NaN,THOUS/MCL,NaN,mmol/L,mmol/L
2,00126ebdaa3f286a,f3ef9a2e287a78e8,mmol/L,MG/DL,mg/dL,%,G/DL,NaN,THOUS/MCL,Unknown,mmol/L,mmol/L
3,0013fc9808033c8f,ef268d8abece3181,mmol/L,NaN,mg/dL,%,G/DL,NaN,THOUS/MCL,NaN,mmol/L,mmol/L
4,001656cc37a919fd,73ae956471950fd0,mmol/L,MG/DL,mg/dL,%,G/DL,NaN,THOUS/MCL,NaN,mmol/L,mmol/L


In [6]:
# check if the units are constant for each lab test

cols_to_check = [col for col in unit_wide.columns if col not in ["LOG_ID", "MRN"]]
unit_check = unit_wide[cols_to_check].nunique(dropna=True)

unit_check

lab
co2           1
crp           1
glucose       2
hematocrit    2
hemoglobin    1
lactate       1
leukocytes    1
pH            1
potassium     1
sodium        1
dtype: int64

In [7]:
# check the different units in glucose
## units are consistent and differ only in capitalization

unit_wide["glucose"].dropna().unique()

array(['mg/dL', 'MG/DL'], dtype=object)

In [8]:
# check the different units in hematocrit

unit_wide["hematocrit"].dropna().unique()

array(['%', 'Unknown'], dtype=object)

In [9]:
# print the rows of unknown units in hematocrit

df[(df["Lab Name"] == "Hematocrit") & (df["Measurement Units"] == "Unknown")]

,LOG_ID,MRN,SMRTDTA_ELEM_VALUE,AGE,HEIGHT,WEIGHT,SEX,AN_START_DATETIME,Lab Code,Lab Name,Observation Value,Measurement Units,Collection Datetime,hypoxemia,bmi,sex,lab
102927,970f45e127d6ddd3,7fb90c1e78deb67a,NaN,90,1.8288,87.845464,Male,2022-11-22 20:10:00,20570-8,Hematocrit,36.2,Unknown,2022-11-22 10:19:00,0,26.265575,1,hematocrit
137371,ca859c70ba34ecdb,b432630c9c3e5f0f,NaN,77,1.6002,44.022631,Female,2018-09-29 12:24:00,20570-8,Hematocrit,37.5,Unknown,2018-09-28 10:07:00,0,17.192042,0,hematocrit


In [10]:
## it seems reasonable to assume that the units are consistent for hematocrit

df[(df["Lab Name"] == "Hematocrit")].head()

,LOG_ID,MRN,SMRTDTA_ELEM_VALUE,AGE,HEIGHT,WEIGHT,SEX,AN_START_DATETIME,Lab Code,Lab Name,Observation Value,Measurement Units,Collection Datetime,hypoxemia,bmi,sex,lab
3,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.6510,65.884098,Male,2021-09-02 11:32:00,20570-8,Hematocrit,27.8,%,2021-09-01 07:50:00,0,24.170546,1,hematocrit
10,0011abe1d4cc9915,c28fd59799fdb891,NaN,69,1.7780,73.806400,Male,2019-06-18 07:09:00,20570-8,Hematocrit,26.7,%,2019-06-18 04:33:00,0,23.346969,1,hematocrit
18,00126ebdaa3f286a,f3ef9a2e287a78e8,NaN,83,1.8034,81.956800,Male,2021-04-23 15:04:00,20570-8,Hematocrit,36.7,%,2021-04-23 08:35:00,0,25.200019,1,hematocrit
26,0013fc9808033c8f,ef268d8abece3181,NaN,79,NaN,87.246353,Male,2019-04-15 14:14:00,20570-8,Hematocrit,39.5,%,2019-04-15 06:44:00,0,NaN,1,hematocrit
34,001656cc37a919fd,73ae956471950fd0,NaN,23,1.5494,86.048414,Female,2022-03-19 10:26:00,20570-8,Hematocrit,38.7,%,2022-03-19 08:02:00,0,35.843942,0,hematocrit


In [11]:
# merge the wide-format lab value table with the original dataset

df_merged = df.merge(value_wide, on=["LOG_ID", "MRN"], how="left")

df_merged.head()

,LOG_ID,MRN,SMRTDTA_ELEM_VALUE,AGE,HEIGHT,WEIGHT,SEX,AN_START_DATETIME,Lab Code,Lab Name,...,co2,crp,glucose,hematocrit,hemoglobin,lactate,leukocytes,pH,potassium,sodium
0,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,1988-5,C reactive protein,...,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0
1,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,2028-9,Carbon dioxide,...,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0
2,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,2345-7,Glucose,...,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0
3,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,20570-8,Hematocrit,...,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0
4,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.651,65.884098,Male,2021-09-02 11:32:00,30350-3,Hemoglobin,...,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0


In [12]:
df_wide = (
    df_merged
    .drop(columns=[
        "Lab Code",
        "Lab Name",
        "Observation Value",
        "Measurement Units",
        "Collection Datetime",
        "lab"
    ])
    .drop_duplicates()
)

df_wide.head()

,LOG_ID,MRN,SMRTDTA_ELEM_VALUE,AGE,HEIGHT,WEIGHT,SEX,AN_START_DATETIME,hypoxemia,bmi,...,co2,crp,glucose,hematocrit,hemoglobin,lactate,leukocytes,pH,potassium,sodium
0,0004678c0906bb9e,6b03d87f76b29311,NaN,70,1.6510,65.884098,Male,2021-09-02 11:32:00,0,24.170546,...,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0
8,0011abe1d4cc9915,c28fd59799fdb891,NaN,69,1.7780,73.806400,Male,2019-06-18 07:09:00,0,23.346969,...,24.0,NaN,121.0,26.7,8.4,NaN,8.0,NaN,4.2,141.0
15,00126ebdaa3f286a,f3ef9a2e287a78e8,NaN,83,1.8034,81.956800,Male,2021-04-23 15:04:00,0,25.200019,...,27.0,15.9,106.0,36.7,12.4,NaN,9.8,7.42,3.7,141.0
24,0013fc9808033c8f,ef268d8abece3181,NaN,79,NaN,87.246353,Male,2019-04-15 14:14:00,0,NaN,...,26.0,NaN,122.0,39.5,14.0,NaN,5.7,NaN,4.5,137.0
31,001656cc37a919fd,73ae956471950fd0,NaN,23,1.5494,86.048414,Female,2022-03-19 10:26:00,0,35.843942,...,23.0,0.3,96.0,38.7,12.9,NaN,8.4,NaN,3.8,138.0


#### Select, reorder and rename the required variables

In [13]:
# keep cols: LOG_ID, MRN, hypoxemia, the 10 lab tests, AGE, sex (binary), BMI, HEIGHT, WEIGHT
cols_id = ["LOG_ID", "MRN"]
cols_outcome = ["hypoxemia"]
cols_lab = [col for col in value_wide.columns if col not in ["LOG_ID", "MRN"]]
cols_demo = ["AGE", "sex", "bmi", "HEIGHT", "WEIGHT"]

cols_keep = cols_id + cols_outcome + cols_lab + cols_demo

df_restructured = df_wide[cols_keep]

df_restructured = df_restructured.rename(columns={
    "AGE": "age",
    "HEIGHT": "height",
    "WEIGHT": "weight"
})

df_restructured.head()

,LOG_ID,MRN,hypoxemia,co2,crp,glucose,hematocrit,hemoglobin,lactate,leukocytes,pH,potassium,sodium,age,sex,bmi,height,weight
0,0004678c0906bb9e,6b03d87f76b29311,0,23.0,7.6,119.0,27.8,9.2,NaN,6.2,NaN,4.4,130.0,70,1,24.170546,1.6510,65.884098
8,0011abe1d4cc9915,c28fd59799fdb891,0,24.0,NaN,121.0,26.7,8.4,NaN,8.0,NaN,4.2,141.0,69,1,23.346969,1.7780,73.806400
15,00126ebdaa3f286a,f3ef9a2e287a78e8,0,27.0,15.9,106.0,36.7,12.4,NaN,9.8,7.42,3.7,141.0,83,1,25.200019,1.8034,81.956800
24,0013fc9808033c8f,ef268d8abece3181,0,26.0,NaN,122.0,39.5,14.0,NaN,5.7,NaN,4.5,137.0,79,1,NaN,NaN,87.246353
31,001656cc37a919fd,73ae956471950fd0,0,23.0,0.3,96.0,38.7,12.9,NaN,8.4,NaN,3.8,138.0,23,0,35.843942,1.5494,86.048414


In [14]:
df_restructured.to_csv("data/restructured_data.csv", index=False)